# Graph Memory — Reasoning Over Relationships, Not Just Similarity

Semantic memory (Demo 02) retrieves by *similarity* but can't reason over *relationships*.
A multi-hop question needs to find an entry point by similarity, then **traverse the graph**.

**The question this demo answers:** *"Who do I know that's connected to flights to Spain?"*

Based on research:
- [GAAMA: Graph Augmented Associative Memory for Agents](https://arxiv.org/abs/2603.27910) — Paul et al., 2026
- [MAGMA: A Multi-Graph based Agentic Memory Architecture for AI Agents](https://arxiv.org/abs/2601.03236) — Jiang et al., 2026
- [GRAVITY: Architecture-Agnostic Structured Anchoring for Long-Horizon Conversational Memory](https://arxiv.org/abs/2605.01688) — Sun et al., 2026

Uses [Strands Agents](https://github.com/strands-agents/sdk-python) for the harness and
[Neo4j](https://neo4j.com/) + [`neo4j-graphrag`](https://neo4j.com/docs/neo4j-graphrag-python/) for graph memory.

## Prerequisites

1. A running **Neo4j** (Desktop, Docker, or Aura).
2. Environment variables in a `.env` file (copy from `.env.example`):
   `OPENAI_API_KEY`, `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`, `NEO4J_DATABASE`.

The demo uses OpenAI by default; swap the model/embeddings for Amazon Bedrock in production (Demo 07).

## Install dependencies

Run this once (or install from a terminal with `uv venv && uv pip install -r requirements.txt`).

In [1]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Configure your model provider

The demo runs with **OpenAI** by default, but you can use **Amazon Bedrock**, **Anthropic**, or any provider available in the Strands configuration — see [supported model providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el).

- **OpenAI (default):** set `OPENAI_API_KEY` below or in a `.env` file. Get one at https://platform.openai.com/api-keys
- **Amazon Bedrock:** no OpenAI key needed — uses your AWS credentials (`aws configure`, with model access enabled in your region). In Test 3's cell (where the chat model is created), comment the `OpenAIModel` lines and uncomment the Bedrock block.

In [2]:
import os

# python-dotenv loads OPENAI_API_KEY and the NEO4J_* connection values from .env,
# so credentials never live inside the notebook.
from dotenv import load_dotenv
load_dotenv()

USING_OPENAI = True  # set False if you switch the chat model to Bedrock in the model cell below
if USING_OPENAI:
    assert os.getenv('OPENAI_API_KEY'), (
        'OPENAI_API_KEY not set — needed for the chat model AND the embeddings in this demo. '
        'Get yours at https://platform.openai.com/api-keys, or switch the chat model to Bedrock below.'
    )
assert os.getenv('NEO4J_PASSWORD'), 'Set NEO4J_* values in your .env'
print('Provider configured')

Provider configured


## Setup — build the graph memory

`graph_memory.build()` connects to Neo4j, ensures an isolated database (and Cypher 25 where needed),
seeds the known graph with real embeddings, and creates the native vector index.

In [3]:
# graph_memory holds all the Neo4j logic for this demo: connect, create the isolated
# database, seed the known graph, build the vector index, and the two retrievers.
import graph_memory as gm

driver, db, embedder = gm.build()
QUESTION = gm.MULTIHOP_QUESTION
print('Question:', QUESTION)

  ✅ Database 'memorydemo' uses Cypher 25 (required by the vector retrievers on this server).


  Seeded graph in database 'memorydemo': 5 nodes, 4 relationships, vector index 'memory_embeddings'.
Question: Who do I know that's connected to flights to Spain?


## The seeded graph

What the agent learned across sessions, stored as connected nodes:

```
(Maya Torres) ─WORKS_AT→ (Iberia) ─MEMBER_OF→ (Oneworld)
                              │
                         FLIES_TO
                              ▼
                          (Madrid) ─IN_COUNTRY→ (Spain)
```

The answer to the question — **Maya Torres** — is never stated; you can only reach it by following edges.

---
## Test 1 — Semantic recall (before)

`VectorRetriever` = pure vector similarity. It surfaces related pieces but can't connect them to a person.

In [4]:
# VectorRetriever (official neo4j-graphrag class) = the 'before': pure vector similarity,
# no traversal — exactly what Demo 02's semantic memory does.
before = gm.make_before_retriever(driver, db, embedder)
result = before.search(query_text=QUESTION, top_k=3)
for item in result.items:
    print(' -', item.content)
print('\nRecovers Maya Torres?', any('Maya Torres' in i.content for i in result.items))

 - {'name': 'Iberia', 'type': 'Airline', 'text': 'Iberia. An airline.'}
 - {'name': 'Spain', 'type': 'Country', 'text': 'Spain. A country.'}
 - {'name': 'Madrid', 'type': 'Location', 'text': 'Madrid. A city.'}

Recovers Maya Torres? False


---
## Test 2 — Graph recall (after)

`VectorCypherRetriever` = similarity to find an entry node, then a Cypher traversal back to the person.
It returns the full chain.

In [5]:
# VectorCypherRetriever = the 'after': vector similarity finds an entry node, then a
# Cypher traversal walks the relationships to the connected person.
after = gm.make_after_retriever(driver, db, embedder)
result = after.search(query_text=QUESTION, top_k=3)
for item in result.items:
    print(' -', item.content)
print('\nRecovers Maya Torres?', any('Maya Torres' in i.content for i in result.items))

 - <Record who='Maya Torres' chain=['Maya Torres', 'Iberia'] score=0.7675593495368958>
 - <Record who='Maya Torres' chain=['Maya Torres', 'Iberia', 'Madrid', 'Spain'] score=0.7081414461135864>
 - <Record who='Maya Torres' chain=['Maya Torres', 'Iberia', 'Madrid'] score=0.6667547225952148>

Recovers Maya Torres? True


---
## Test 3 — A full Strands agent with graph memory

The agent uses `recall_graph` to answer, and `remember_fact` to write a new fact back into the graph —
the harness is just tools + state.

In [6]:
# Tests 1-2 needed no chat model — retrievers talk to Neo4j directly. An AGENT does:
# Agent is the Strands agent loop, and this is where the language model enters.
from strands import Agent

# OTEL_SDK_DISABLED silences OpenTelemetry tracing noise from the agent's output.
os.environ['OTEL_SDK_DISABLED'] = 'true'

# Using OpenAI-compatible interface via Strands SDK (not direct OpenAI usage)
from strands.models.openai import OpenAIModel

MODEL = OpenAIModel(model_id='gpt-4o-mini')  # api_key read from the OPENAI_API_KEY env var

# To run the CHAT model on Amazon Bedrock instead (embeddings stay OpenAI for now),
# comment the two lines above and uncomment these two:
# from strands.models import BedrockModel
# MODEL = BedrockModel(model_id='openai.gpt-oss-120b-1:0', region_name='us-west-2')

# travel_tools wraps the two retrievers plus a write tool (remember_fact) as @tool
# functions the agent can call.
import travel_tools as tt

tt.init_memory(driver=driver, db=db, embedder=embedder)

agent = Agent(
    model=MODEL,
    system_prompt=(
        'You are a travel assistant with graph memory. Use recall_graph to answer '
        'questions about people and places, and remember_fact to store new durable facts. Be concise.'
    ),
    tools=[tt.recall_graph, tt.recall_semantic, tt.remember_fact],
    callback_handler=None,
)

resp = agent(QUESTION)
print('Agent:', resp.message['content'][0]['text'].strip())

Agent: You know Maya Torres, who is connected to flights to Spain, specifically with Iberia to Madrid.


In [7]:
resp = agent('By the way, remember that Maya Torres works at Iberia — she is my contact there.')
print('Agent:', resp.message['content'][0]['text'].strip())
print('\nFacts logged to agent.state:', agent.state.get('remembered_facts'))

Agent: Got it! I've noted that Maya Torres works at Iberia and is your contact there.

Facts logged to agent.state: [{'subject': 'Maya Torres', 'relation': 'WORKS_AT', 'object': 'Iberia'}]


---
## Test 4 — Deterministic scorecard

Four multi-hop questions, checked against the known graph (no LLM judge). Reproducible.

In [8]:
SCORECARD = [
    ("Who do I know that's connected to flights to Spain?", 'Maya Torres'),
    ('Who do I know connected to an airline that flies to Madrid?', 'Maya Torres'),
    ('Who works at the Oneworld airline I know?', 'Maya Torres'),
    ('Which person is linked to airlines in Spain?', 'Maya Torres'),
]

before_hits = after_hits = 0
print(f"{'Question':<52}{'before':>8}{'after':>8}")
for q, target in SCORECARD:
    b = any(target in it.content for it in before.search(query_text=q, top_k=3).items)
    a = any(target in it.content for it in after.search(query_text=q, top_k=3).items)
    before_hits += b; after_hits += a
    print(f"{q[:52]:<52}{('OK' if b else '-'):>8}{('OK' if a else '-'):>8}")

print(f'\nCorrect — before: {before_hits}/{len(SCORECARD)} | after: {after_hits}/{len(SCORECARD)}')

Question                                              before   after


Who do I know that's connected to flights to Spain?        -      OK


Who do I know connected to an airline that flies to        -      OK


Who works at the Oneworld airline I know?                 OK      OK


Which person is linked to airlines in Spain?               -      OK

Correct — before: 1/4 | after: 4/4


---
## Summary

| Retriever | Strategy | Multi-hop answer? |
|-----------|----------|-------------------|
| `VectorRetriever` (before) | Pure similarity | No — finds pieces, can't connect them |
| `VectorCypherRetriever` (after) | Similarity + traversal | Yes — returns the full chain |

**Key insight:** similarity finds related pieces; only traversal connects them. Graph memory answers
multi-hop questions that flat/semantic memory cannot — and with Strands, plugging in the graph store
is just tools + state.

Both retrievers got the **same facts** and shared the **same vector index**. The graph wins structurally,
not because it was handed the answer.

In [9]:
driver.close()
print('Done.')

Done.
